# 第3章 金融数据获取与清洗 —— 配套代码

对应正文 `book/part1/03-data-acquisition.md`。清洗部分用内置数据离线可跑；
最后一格「联网抓取」需 `uv sync --extra data`，默认不执行。

## 模拟停牌缺失与对齐

In [ ]:
import numpy as np
import pandas as pd
from fds import load_sample_prices

prices = load_sample_prices()

# 人为制造 TECH 的若干「停牌」缺失
dirty = prices.copy()
rng = np.random.default_rng(42)
halt_idx = rng.choice(len(dirty), size=10, replace=False)
dirty.iloc[halt_idx, dirty.columns.get_loc("TECH")] = np.nan
print("TECH 缺失天数：", int(dirty["TECH"].isna().sum()))

In [ ]:
# 前向填充：停牌日沿用上一交易日价格
clean = dirty.ffill()
print("填充后缺失天数：", int(clean["TECH"].isna().sum()))

## 异常值识别：用涨跌幅约束

A 股主板日涨跌幅一般在 ±10% 内，超出即可疑（这里数据为合成，仅演示方法）。

In [ ]:
ret = clean.pct_change()
suspicious = (ret.abs() > 0.10).sum()
print("各股票疑似异常日收益次数（|日涨跌|>10%）：")
print(suspicious)

## 多标的对齐示意

把不同起点的序列按日期索引对齐（concat 自动对齐），再排序、前向填充。

In [ ]:
s1 = clean["BANK"].iloc[50:]      # 较晚「上市」
s2 = clean["LIQUOR"]
panel = pd.concat([s1, s2], axis=1).sort_index().ffill()
panel.head()

## （可选·联网）用 akshare 抓取真实 A 股

需先 `uv sync --extra data`。下方用 try/except 包裹，未联网或未装包时不报错。

In [ ]:
try:
    import akshare as ak
    real = ak.stock_zh_a_hist(symbol="600519", period="daily",
                              start_date="20240101", end_date="20241231", adjust="qfq")
    print(real.tail())
except Exception as e:
    print("未抓取（未装 akshare 或无网络），跳过：", type(e).__name__)